## 2751. Robot Collisions (hard)

There are n 1-indexed robots, each having a position on a line, health, and movement direction.

You are given 0-indexed integer arrays positions, healths, and a string directions (directions[i] is either 'L' for left or 'R' for right). All integers in positions are unique.

All robots start moving on the line simultaneously at the same speed in their given directions. If two robots ever share the same position while moving, they will collide.

If two robots collide, the robot with lower health is removed from the line, and the health of the other robot decreases by one. The surviving robot continues in the same direction it was going. If both robots have the same health, they are both removed from the line.

Your task is to determine the health of the robots that survive the collisions, in the same order that the robots were given, i.e. final health of robot 1 (if survived), final health of robot 2 (if survived), and so on. If there are no survivors, return an empty array.

Return an array containing the health of the remaining robots (in the order they were given in the input), after no further collisions can occur.

Note: The positions may be unsorted.

Example

Input: positions = [5,4,3,2,1], healths = [2,17,9,15,10], directions = "RRRRR"

Output: [2,17,9,15,10]

Explanation: 

No collision occurs in this example, since all robots are moving in the same direction. So, the health of the robots in order from the first robot is returned, [2, 17, 9, 15, 10].

### 1. First Version (using deque)

In [ ]:
from collections import deque
class Solution:
    def survivedRobotsHealths(self, positions: List[int], healths: List[int], directions: str) -> List[int]:
        n = len(positions)
        robots = []
        for i in range(n):
            robots.append([positions[i], healths[i], directions[i], i])
        
        # 依第一個元素由小到大排序
        robots.sort(key = lambda x: x[0])

        pass_robots = []
        robot_to_right = deque() # 方向向右的right

        for robot in robots: 
            if robot[2] == 'L':
                if not robot_to_right:
                    pass_robots.append(robot) # 活下來了
                    continue
                while robot_to_right:
                    s_robot = robot_to_right.pop()
                    if s_robot[1] > robot[1]: # s_robot獲勝
                        s_robot[1] -= 1
                        if s_robot[1] > 0: robot_to_right.append(s_robot) # 若s_robot扣完血還活著才要存回stack
                        break
                    elif s_robot[1] == robot[1]:
                        break
                    else: # s_robot輸了
                        robot[1] -= 1
                        if robot[1] == 0: break
                        if not robot_to_right: pass_robots.append(robot) # 沒有對手了
            else:
                robot_to_right.append(robot)

        all_survivors = list(robot_to_right) + pass_robots
        all_survivors.sort(key = lambda x: x[3]) # 照機器人編號排好
        
        output = []
        for survivor in all_survivors:
            output.append(survivor[1])
        return output

### 2. 只用list來當stack效率更好

In [ ]:
class Solution:
    def survivedRobotsHealths(self, positions: List[int], healths: List[int], directions: str) -> List[int]:
        n = len(positions)

        # 只對index依position做排序，節省時間
        indices = sorted(range(n), key = lambda i: positions[i])

        stack = [] # 用list當stack效率最高

        for i in indices:
            if directions[i] == 'R':
                stack.append(i)
            else:
                while stack and healths[i] > 0:
                    top_i = stack[-1]
                    if healths[top_i] > healths[i]:
                        healths[top_i] -= 1
                        healths[i] = 0
                    elif healths[top_i] < healths[i]:
                        healths[top_i] = 0
                        healths[i] -= 1
                        stack.pop()  # list pop 為 O(1)
                    else:
                        healths[top_i] = 0
                        healths[i] = 0
                        stack.pop()
        
        return [h for h in healths if h > 0]